In [ ]:
from pathlib import Path

from openai import OpenAI

from dotenv import load_dotenv

import datetime

load_dotenv()

client = OpenAI()

# Where the pyproject.toml file is located.
BASE_DIR = Path().cwd().parent


INPUT_DIR = BASE_DIR.joinpath("part_a", "data", "titulos_codigo_mineria")

OUTPUT_DIR = BASE_DIR.joinpath("part_a", "data")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

QUESTIONS_DIR = OUTPUT_DIR.joinpath("generated_questions")
QUESTIONS_DIR.mkdir(exist_ok=True, parents=True)

def prompt_model(system_prompt, model, max_tokens=1_000, temperature=1.0, top_p=1.0):
    """
    Generate questions based on the provided normalized text.

    Args:
        system_prompt (str): The system prompt to guide the question generation.
        model (str): The OpenAI model to use for generating questions.
        max_tokens (int): The maximum number of tokens to generate.
        temperature (float): The temperature for the generation.
        top_p (float): The top-p value for nucleus sampling.
    Returns:
        str: The generated questions in markdown format.
    """

    response = client.responses.create(
    model=model,
    input=[
        {
        "role": "system",
        "content": [
            {
            "type": "input_text",
            "text": system_prompt,
            }
        ]
        }
    ],
    text={
        "format": {
        "type": "text"
        }
    },
    reasoning={},
    tools=[],
    temperature=temperature,
    max_output_tokens=max_tokens,
    top_p=top_p,
    store=True # Always store the response
    )

    return response

def generate_questions(system_prompt, system_prompt_version, model, timestamp, input_dir, output_dir, max_tokens=1_000, temperature=1.0, top_p=1.0):
    """
    Generate questions based on the provided normalized text.

    Args:
        model (str): The OpenAI model to use for generating questions.
        max_tokens (int): The maximum number of tokens to generate.
        temperature (float): The temperature for the generation.
        top_p (float): The top-p value for nucleus sampling.
    Returns:
        str: The generated questions in markdown format.
    """

    for norm_text_path in input_dir.glob("*.md"):
        with open(norm_text_path, "r", encoding="utf-8") as file:
            norm_text = file.read()

        response = prompt_model(
            system_prompt=system_prompt.format(norm_text=norm_text),
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
        )

        response_model = response.model
        source_file = norm_text_path.stem

        questions_output_file = output_dir.joinpath(f"{timestamp}_{response_model}_{system_prompt_version}_{source_file}.md")
        model_response_file = output_dir.joinpath(f"{timestamp}_{response_model}_{system_prompt_version}_{source_file}.json")

        response.to_dict()["output"][0]["content"][0]["text"]

        with open(questions_output_file, "w", encoding="utf-8") as file:
            file.write(response.to_dict()["output"][0]["content"][0]["text"])
        with open(model_response_file, "w", encoding="utf-8") as file:
            file.write(response.to_json())

In [49]:
now = datetime.datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H%M%S")
model = "gpt-4.1"
max_tokens = 10_000
temperature = 1.0
top_p = 1.0
system_prompt_version = "v1"
system_prompt = """
Sos un abogado experto en el código de minería argentino que dicta una especialización para otros abogados.

Usando el texto adjunto como base:
 - Crea una pregunta comprehensiva (puede contener subitems) para evaluar el conocimientos relativo al código de minería argentino de los abogados que cursan la especialización.
 - La pregunta debe incluir una respuesta correcta de ejemplo.
 - La pregunta debe ser planteada como un caso a resolver.
 - La pregunta no debe ser de tipo "multiple choice".
 - Responde solo en español.
 - Responde con la pregunta y la respuesta en formato markdown. No añadas texto como "Claro, aquí tienes una pregunta..."

Texto adjunto:
{norm_text}
"""

generate_questions(
    system_prompt=system_prompt,
    system_prompt_version=system_prompt_version,
    model=model,
    timestamp=timestamp,
    input_dir=INPUT_DIR,
    output_dir=QUESTIONS_DIR,
    max_tokens=max_tokens,
    temperature=temperature,
    top_p=top_p
)
